In [1]:
from sklearn.cluster import DBSCAN
from math import sqrt
from scipy.stats import norm
from scipy.stats import binom
import numpy as np
import pandas as pd
import os
from utils import *

In [2]:
AllFilePath = r"C:\Users\Andrew\Documents\Acoustic-Space-Boiling\src\labels.csv"
NewFilePath = r"C:\Users\Andrew\Documents\Acoustic-Space-Boiling\src\parsed_labels.csv"

AllDataSet = pd.read_csv(AllFilePath)
NewDataSet = pd.read_csv(NewFilePath)

ParsedLabels = NewDataSet['filename']

ParsedLabels = [s.replace('.png', '').replace('MATLAB ', '') for s in ParsedLabels]
# print(ParsedLabels)

NewDataSet['filename'] = ParsedLabels

IntersectingLabels = list(set(ParsedLabels) & set(AllDataSet['file_name']))

FinalLabels = {
    'filename': [],
    'label': []
    }

for index, row in AllDataSet.iterrows():
    if row['file_name'] in IntersectingLabels:
        label = NewDataSet.loc[NewDataSet['filename'] == row['file_name']]['label']
        label = int(label.iloc[0])
        FinalLabels['label'].append(label)
    else:
        FinalLabels['label'].append(row['label'])
    FinalLabels['filename'].append(row['file_name'])
    
print(len(FinalLabels['filename']))
print(len(FinalLabels['label']))

FinalLabels = pd.DataFrame(FinalLabels)

FinalLabels.to_csv(r'C:\Users\Andrew\Documents\Acoustic-Space-Boiling\src\finallabels.csv', index= False)


288
288


1	Single Rhythmic
2	Double Rhythmic
3	Random
4	Rhythmic with Climax
5	Noise
6	1 Rhythmic with Random
7	Triple Rhythmic
8	Transition

In [5]:
# Loop through and create DF
CSVPaths = r'C:\Users\Andrew\Documents\Acoustic-Space-Boiling\Data\After_May'

rows = []
for File in os.listdir(CSVPaths):
    full = os.path.join(CSVPaths, File)
    acceleration0 = get_file(full)
    x, _ = get_peaks(acceleration0)

    if len(x) > 3:
        diffs = get_diffs(x)
    else:
        diffs = [0]

    # look up your label for this file:
    CleanedFile = File.replace('.png', '').replace('MATLAB ', '')
    # print(CleanedFile)
    lbl = FinalLabels.loc[FinalLabels['filename'] == CleanedFile, 'label'].iat[0]

    rows.append({
        'file_name': File,
        'diffs': diffs,
        'label': int(lbl)
    })
    


df = pd.DataFrame(rows)

mapping = {
    1: 1,
    2: 2,
    3: 1,
    4: 1,
    5: 0,
    6: 2,
    7: 3,
    8: 2,
}

df['label'] = df['label'].map(mapping)

In [22]:
import numpy as np
import pandas as pd

# 1. Define binning parameters
bin_width   = 0.01
max_period  = 10.0
num_bins    = int(max_period / bin_width)   # => 1000
# bin centers 0.00, 0.01, …, 9.99
bin_centers = np.arange(0, max_period, bin_width)
# column names with two decimals
bin_cols    = [f"{c:.2f}s" for c in bin_centers]

# histogram edges for numpy.histogram
edges = np.arange(0, max_period + bin_width, bin_width)  # 0.00 to 10.00

# 2. Build the (n_samples × num_bins) matrix
hist_matrix = np.vstack([
    np.histogram(
        [d for d in dlist if 0 <= d < max_period],
        bins=edges
    )[0]
    for dlist in df['diffs']
])

# 3. Turn it into a DataFrame
hist_df = pd.DataFrame(hist_matrix, columns=bin_cols)
hist_df = hist_df.astype(int)

# 4. Grab your metadata, reset its index to align by row
meta = df[['file_name','label']].reset_index(drop=True)

# 5. Concatenate side-by-side
df_feat = pd.concat([meta, hist_df], axis=1)

print(df_feat)


                                      file_name  label  0.00s  0.01s  0.02s  \
0    MATLAB 1-00 PM Fri, Jun 28, 2024 Run8 .csv      2      0      0      0   
1     MATLAB 1-02 PM Thu, Nov 7, 2024 Run8 .csv      1      0      0      0   
2     MATLAB 1-04 PM Thu, Aug 22, 2024 Run2.csv      1      0      0      0   
3    MATLAB 1-04 PM Tue, Sep 10, 2024 Run9 .csv      1      0      0      0   
4    MATLAB 1-05 PM Thu, Aug 22, 2024 Run3 .csv      1      0      0      0   
..                                          ...    ...    ...    ...    ...   
283  MATLAB 4-58 PM Thu, Oct 10, 2024 Run12.csv      1      0      0      0   
284  MATLAB 4-59 PM Tue, Oct 1, 2024 Run12 .csv      1      0      0      0   
285  MATLAB 5-01 PM Thu, Oct 10, 2024 Run13.csv      2      0      0      0   
286  MATLAB 5-01 PM Tue, Oct 1, 2024 Run13 .csv      1      1      0      0   
287  MATLAB 5-05 PM Thu, Oct 10, 2024 Run14.csv      1      0      0      0   

     0.03s  0.04s  0.05s  0.06s  0.07s  ...  9.90s 

In [24]:
top_n = 3
for idx, row in df_feat.iterrows():
    fname = row['file_name']
    lbl   = row['label']
    # force the histogram slice to ints
    hist  = row[bin_cols].astype(int)

    top_bins = hist.nlargest(top_n)
    print(f"File: {fname}, label: {lbl}")
    for bin_name, count in top_bins.items():
        print(f"  {bin_name}: {count}")
    print('-'*40)


File: MATLAB 1-00 PM Fri, Jun 28, 2024 Run8 .csv, label: 2
  0.91s: 7
  1.22s: 7
  2.75s: 6
----------------------------------------
File: MATLAB 1-02 PM Thu, Nov 7, 2024 Run8 .csv, label: 1
  0.47s: 18
  0.23s: 16
  0.70s: 15
----------------------------------------
File: MATLAB 1-04 PM Thu, Aug 22, 2024 Run2.csv, label: 1
  1.24s: 2
  1.51s: 2
  2.20s: 2
----------------------------------------
File: MATLAB 1-04 PM Tue, Sep 10, 2024 Run9 .csv, label: 1
  9.52s: 2
  0.57s: 1
  3.73s: 1
----------------------------------------
File: MATLAB 1-05 PM Thu, Aug 22, 2024 Run3 .csv, label: 1
  2.51s: 2
  2.70s: 2
  2.83s: 2
----------------------------------------
File: MATLAB 1-07 PM Fri, Jun 14, 2024 Run1 .csv, label: 1
  0.52s: 2
  2.59s: 2
  3.59s: 2
----------------------------------------
File: MATLAB 1-10 PM Thu, Aug 22, 2024 Run4 .csv, label: 1
  3.35s: 3
  1.81s: 2
  4.06s: 2
----------------------------------------
File: MATLAB 1-12 PM Thu, Nov 7, 2024 Run9 .csv, label: 1
  0.21s: 7

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = df_feat[bin_cols].values
X_scaled = scaler.fit_transform(X)
y = df_feat['label'].values


In [26]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, accuracy_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y), 1):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    clf = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        max_iter=200,
        random_state=42,
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)

    # overall accuracy
    acc = accuracy_score(y_val, y_pred)
    print(f"\nFold {fold} — overall accuracy: {acc:.3f}\n")

    # per-class precision / recall / f1 (recall ≡ per-class accuracy)
    print(classification_report(y_val, y_pred, zero_division=0))


c:\Users\Andrew\Documents\Acoustic-Space-Boiling\.venv\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(



Fold 1 — overall accuracy: 0.586

              precision    recall  f1-score   support

           0       0.42      1.00      0.59         8
           1       0.76      0.50      0.60        32
           2       0.56      0.59      0.57        17
           3       0.00      0.00      0.00         1

    accuracy                           0.59        58
   macro avg       0.43      0.52      0.44        58
weighted avg       0.64      0.59      0.58        58


Fold 2 — overall accuracy: 0.621

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         8
           1       0.60      0.91      0.72        32
           2       0.70      0.41      0.52        17
           3       0.00      0.00      0.00         1

    accuracy                           0.62        58
   macro avg       0.33      0.33      0.31        58
weighted avg       0.54      0.62      0.55        58


Fold 3 — overall accuracy: 0.534

              precision  